In [5]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import io
import cv2
import numpy as np
from PIL import Image
from skimage.metrics import structural_similarity as ssim

def load_clip():
    try:
        import torch
        from transformers import CLIPProcessor, CLIPModel
        model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        model.eval()
        return model, processor
    except Exception as e:
        return None, None

def read_cv(pil):
    rgb = np.array(pil.convert("RGB"))
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

def resize_pair(a, b, size=(512, 512)):
    a = cv2.resize(a, size, interpolation=cv2.INTER_AREA)
    b = cv2.resize(b, size, interpolation=cv2.INTER_AREA)
    return a, b

def structural_score(a, b):
    a, b = resize_pair(a, b)
    ga = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY)
    gb = cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    return float(ssim(ga, gb, data_range=255)) * 100

def color_score(a, b):
    a, b = resize_pair(a, b)
    ah = cv2.calcHist([a], [0,1], None, [32,32], [0,256,0,256])
    bh = cv2.calcHist([b], [0,1], None, [32,32], [0,256,0,256])
    cv2.normalize(ah, ah)
    cv2.normalize(bh, bh)
    corr = cv2.compareHist(ah, bh, cv2.HISTCMP_CORREL)
    return float(np.clip((corr + 1) / 2, 0, 1) * 100)

def feature_score(a, b):
    a, b = resize_pair(a, b)
    g1 = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY)
    g2 = cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(nfeatures=1200)
    k1, d1 = orb.detectAndCompute(g1, None)
    k2, d2 = orb.detectAndCompute(g2, None)
    if d1 is None or d2 is None or len(k1) < 2 or len(k2) < 2:
        return 0.0, 0
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
    matches = matcher.knnMatch(d1, d2, k=2)
    good = [m for pair in matches if len(pair) == 2 for m, n in [pair] if m.distance < 0.72*n.distance]
    # Saturating score: enough strong matches gives a high structural/feature score.
    score = 100 * (1 - np.exp(-len(good) / 35))
    return float(np.clip(score, 0, 100)), len(good)

def clip_score(pil_a, pil_b):
    model, processor = load_clip()
    if model is None:
        return None
    import torch
    inputs = processor(images=[pil_a.convert("RGB"), pil_b.convert("RGB")], return_tensors="pt")
    with torch.no_grad():
        feats = model.get_image_features(**inputs).pooler_output
        feats = feats / feats.norm(dim=-1, keepdim=True)
        sim = float((feats[0] @ feats[1]).item())
    # CLIP cosine similarity typically lies around 0..1 for related images.
    return float(np.clip(sim * 100, 0, 100))

def overall(semantic, structural, color, feature):
    # Semantic gets the largest weight because generated posters can look different
    # while depicting the same concept. Feature/structure helps detect composition.
    if semantic is None:
        return 0.45*structural + 0.20*color + 0.35*feature
    return 0.50*semantic + 0.20*structural + 0.10*color + 0.20*feature

def verdict(score):
    if score >= 85: return "Very high similarity"
    if score >= 70: return "High similarity"
    if score >= 55: return "Moderate similarity"
    if score >= 40: return "Low similarity"
    return "Very low similarity"

In [3]:
def pipeline(original, poster):
    original = Image.open(original)
    poster = Image.open(poster)
    a = read_cv(original)
    b = read_cv(poster)
    s_struct = structural_score(a, b)
    s_color = color_score(a, b)
    s_feature, match_count = feature_score(a, b)
    s_semantic = clip_score(original, poster)
    final = overall(s_semantic, s_struct, s_color, s_feature)
    return final

In [9]:
import os

BASE_PATH = "/content/drive/MyDrive/Round 3 (File responses)"

ref_path = os.path.join(BASE_PATH, "ref.jpeg")
teams_folder = os.path.join(BASE_PATH, "Image (File responses)")

results = []

for filename in os.listdir(teams_folder):

    file_path = os.path.join(teams_folder, filename)

    # Skip folders and non-image files
    if not os.path.isfile(file_path):
        continue

    if not filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
        continue

    try:
        score = pipeline(ref_path, file_path)

        results.append({
            "team": filename,
            "score": score
        })

        print(f"{filename}: {score:.2f}")

    except Exception as e:
        print(f"{filename}: ERROR - {e}")


# Sort by score, highest first
results.sort(key=lambda x: x["score"], reverse=True)

print("\n===== FINAL RANKING =====")

for rank, result in enumerate(results, start=1):
    print(f"{rank}. {result['team']} -> {result['score']:.2f}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

123.png: 90.97


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

345.jpeg: 100.00

===== FINAL RANKING =====
1. 345.jpeg -> 100.00
2. 123.png -> 90.97
